[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/iJO1366_Escher_flux_exercise.ipynb)

# iJO1366 Flux Balance Analysis and Escher Visualization

This notebook is a hands-on exercise in **constraint-based modeling** using the genome-scale metabolic model of *Escherichia coli* K-12 MG1655, **iJO1366** (BiGG database).

You will:
1. Load the COBRA JSON model and inspect the growth objective.
2. Simulate growth on glucose or glycerol under aerobic or anaerobic conditions.
3. Compare predicted growth rates and metabolic fluxes.
4. Export flux data for visualization in **[Escher](https://escher.github.io/)**.
5. Predict the effect of reaction knockouts.

> **Note:** We use **Escher** (metabolic map visualization), not Etcher.

---

## Student exercises

### Exercise 1 — Load iJO1366 and inspect the objective
Run the setup cells below. Load the model and identify the **biomass / growth objective reaction**. How many reactions and metabolites does the model contain?

### Exercise 2 — Glucose aerobic vs anaerobic growth
Simulate wild-type growth on glucose with oxygen (`glucose_aerobic`) and without oxygen (`glucose_anaerobic`). Record the predicted growth rate (objective value).

### Exercise 3 — Glycerol aerobic vs anaerobic growth
Repeat Exercise 2 using glycerol as the sole carbon source.

### Exercise 4 — Compare growth rates and explain differences
Use `outputs/growth_summary.csv` to compare conditions. Why is aerobic growth typically faster than anaerobic? Why does glucose support higher growth than glycerol?

### Exercise 5 — Visualize fluxes in Escher
Open [https://escher.github.io/](https://escher.github.io/), load the iJO1366 model and central metabolism map, then import one of the exported CSV files via **Data → Load reaction data**.

### Exercise 6 — Reaction knockouts under glucose aerobic medium
Simulate knockouts of **PFK**, **G6PDH2r**, **CS**, and **ATPS4rpp** (when present in the model). Compare mutant growth rates and fluxes to wild-type `glucose_aerobic`.

### Exercise 7 — Identify reactions with the largest flux changes
Inspect `outputs/top_flux_changes.csv`. Which pathways show the biggest flux redistribution between conditions or after knockouts? Relate your findings to biochemistry.

## 0. Install packages (Google Colab)

Run this cell first in a **clean Colab session**. All dependencies are installed here — nothing needs to be pre-installed.

In [ ]:
import sys

# GLPK solver support (used by COBRApy / optlang)
!apt-get -qq update -y
!apt-get -qq install -y swig libgmp-dev 2>/dev/null

!{sys.executable} -m pip install -q cobra escher pandas numpy matplotlib ipywidgets swiglpk

## 1. Imports and folder setup

In [ ]:
import os
import shutil
import warnings
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

import cobra
import escher
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("COBRApy version:", cobra.__version__)
print("Escher version:", escher.__version__)

# Create working folders
for folder in ["data", "outputs", "escher_outputs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"Created/verified folder: {folder}/")

## 2. Download the iJO1366 model

In [ ]:
MODEL_PATH = Path("data/iJO1366.json")
PRIMARY_URL = "http://bigg.ucsd.edu/static/models/iJO1366.json"
FALLBACK_URL = "http://bigg.ucsd.edu/api/v2/models/iJO1366/download"


def download_model(output_path=MODEL_PATH):
    """Download iJO1366 JSON from BiGG (primary URL, then fallback)."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    for url in [PRIMARY_URL, FALLBACK_URL]:
        try:
            print(f"Trying: {url}")
            with urlopen(url, timeout=120) as response:
                data = response.read()
            if len(data) < 1000:
                raise ValueError("Downloaded file is unexpectedly small.")
            output_path.write_bytes(data)
            print(f"Saved model to {output_path} ({len(data):,} bytes)")
            return output_path
        except (URLError, HTTPError, ValueError, TimeoutError) as exc:
            print(f"  Failed: {exc}")

    raise RuntimeError("Could not download iJO1366 from BiGG. Check your internet connection.")


if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1000:
    download_model()
else:
    print(f"Model already present at {MODEL_PATH}")

## 3. Reusable functions

The functions below keep the workflow modular and safe: each simulation uses a **context manager** (`with model:`) so the base model is not permanently modified.

In [ ]:
CARBON_SOURCES = {
    "glucose": "EX_glc__D_e",
    "glycerol": "EX_glyc_e",
}

KEY_EXCHANGES = [
    "EX_glc__D_e",
    "EX_glyc_e",
    "EX_o2_e",
    "EX_nh4_e",
    "EX_pi_e",
    "EX_so4_e",
]

INORGANIC_EXCHANGES = [
    "EX_nh4_e", "EX_pi_e", "EX_so4_e", "EX_k_e", "EX_na1_e", "EX_cl_e",
    "EX_mg2_e", "EX_ca2_e", "EX_fe2_e", "EX_fe3_e", "EX_h2o_e", "EX_h_e",
]

MUTANT_REACTIONS = ["PFK", "G6PDH2r", "CS", "ATPS4rpp"]


def load_model(model_path=MODEL_PATH):
    """Load the iJO1366 model from a COBRA JSON file."""
    model_path = Path(model_path)
    if not model_path.exists():
        download_model(model_path)
    model = cobra.io.load_json_model(str(model_path))
    return model


def validate_reactions(model, reaction_ids):
    """Return (present, missing) lists for a set of reaction IDs."""
    present, missing = [], []
    for reaction_id in reaction_ids:
        if reaction_id in model.reactions:
            present.append(reaction_id)
        else:
            missing.append(reaction_id)
    return present, missing


def get_objective_reaction_id(model):
    """Detect the active biomass/objective reaction from model coefficients."""
    objective_reactions = [
        rxn.id for rxn in model.reactions if rxn.objective_coefficient != 0
    ]
    if not objective_reactions:
        raise ValueError("No objective reaction found in the model.")
    if len(objective_reactions) > 1:
        print("Warning: multiple objective reactions detected:", objective_reactions)
    return objective_reactions[0]


def set_medium(model, carbon_source, carbon_uptake=10, oxygen_uptake=20):
    """Configure an M9-like minimal medium on the model.

    Parameters
    ----------
    carbon_source : str
        Exchange reaction ID for the sole carbon source (e.g. EX_glc__D_e).
    carbon_uptake : float
        Carbon uptake rate (mmol/gDW/h). Default 10.
    oxygen_uptake : float
        O2 uptake rate. Use 20 for aerobic, 0 for anaerobic.
    """
    medium = {}

    # Open inorganic nutrients that exist in the model
    for ex_id in INORGANIC_EXCHANGES:
        if ex_id in model.reactions:
            medium[ex_id] = 1000.0

    # Carbon source uptake (positive value = allowed uptake in COBRApy medium dict)
    if carbon_source in model.reactions:
        medium[carbon_source] = float(carbon_uptake)
    else:
        warnings.warn(f"Carbon source exchange {carbon_source} not found in model.")

    # Oxygen
    if "EX_o2_e" in model.reactions:
        medium["EX_o2_e"] = float(oxygen_uptake)
    elif oxygen_uptake > 0:
        warnings.warn("EX_o2_e not found; cannot set aerobic conditions explicitly.")

    model.medium = medium
    return model


def run_fba(model, condition_name, carbon_source=None, oxygen_uptake=None, knockout=None):
    """Run FBA and return a results dictionary."""
    solution = model.optimize()
    status = solution.status
    objective_value = solution.objective_value

    if status != "optimal":
        print(
            f"WARNING [{condition_name}]: solver status = '{status}' "
            f"(objective = {objective_value})"
        )

    return {
        "condition": condition_name,
        "status": status,
        "objective_value": objective_value,
        "fluxes": solution.fluxes,
        "carbon_source": carbon_source,
        "oxygen_uptake": oxygen_uptake,
        "knockout": knockout,
    }


def apply_reaction_knockout(model, reaction_id):
    """Knock out a reaction if it exists. Returns (success, message)."""
    if reaction_id not in model.reactions:
        return False, f"Reaction '{reaction_id}' not found — skipping knockout."
    model.reactions.get_by_id(reaction_id).knock_out()
    return True, f"Knocked out '{reaction_id}'."


def export_fluxes_for_escher(results_dict, output_dir="outputs"):
    """Export Escher-compatible CSV files from a dict of FBA results."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Collect all reaction IDs across conditions
    all_reactions = set()
    for result in results_dict.values():
        all_reactions.update(result["fluxes"].index)
    all_reactions = sorted(all_reactions)

    # Wide format: reaction + one column per condition
    wide_data = {"reaction": all_reactions}
    for condition_name, result in results_dict.items():
        wide_data[condition_name] = [
            result["fluxes"].get(rxn, 0.0) for rxn in all_reactions
        ]
    wide_df = pd.DataFrame(wide_data)
    wide_path = output_dir / "escher_reaction_fluxes_wide.csv"
    wide_df.to_csv(wide_path, index=False)
    print(f"Saved {wide_path}")

    # Comparison tables for Escher
    comparisons = {
        "escher_glucose_aerobic_vs_anaerobic.csv": (
            "glucose_aerobic", "glucose_anaerobic"
        ),
        "escher_glucose_vs_glycerol_aerobic.csv": (
            "glucose_aerobic", "glycerol_aerobic"
        ),
    }

    for filename, (col_a, col_b) in comparisons.items():
        if col_a in results_dict and col_b in results_dict:
            comp_df = pd.DataFrame({
                "reaction": all_reactions,
                col_a: [results_dict[col_a]["fluxes"].get(r, 0.0) for r in all_reactions],
                col_b: [results_dict[col_b]["fluxes"].get(r, 0.0) for r in all_reactions],
            })
            comp_df.to_csv(output_dir / filename, index=False)
            print(f"Saved {output_dir / filename}")
        else:
            print(f"Skipping {filename}: missing one or both conditions.")

    # Mutant comparison (glucose aerobic wild-type vs knockouts)
    mutant_cols = ["glucose_aerobic"]
    for rxn_id in MUTANT_REACTIONS:
        col_name = f"mutant_{rxn_id}"
        if col_name in results_dict:
            mutant_cols.append(col_name)

    if len(mutant_cols) > 1:
        mutant_data = {"reaction": all_reactions}
        for col in mutant_cols:
            mutant_data[col] = [
                results_dict[col]["fluxes"].get(r, 0.0) for r in all_reactions
            ]
        mutant_df = pd.DataFrame(mutant_data)
        mutant_path = output_dir / "escher_mutants_glucose_aerobic.csv"
        mutant_df.to_csv(mutant_path, index=False)
        print(f"Saved {mutant_path}")

    return wide_df


def make_escher_html(model_json_path, flux_csv_path, output_html_path, condition_column=None):
    """Try to build an Escher HTML map with flux data. Fail gracefully in Colab."""
    model_json_path = str(model_json_path)
    flux_csv_path = str(flux_csv_path)
    output_html_path = str(output_html_path)

    try:
        flux_df = pd.read_csv(flux_csv_path)
        if condition_column is None:
            # Pick the first non-reaction column
            condition_column = [c for c in flux_df.columns if c != "reaction"][0]

        reaction_data = dict(zip(flux_df["reaction"], flux_df[condition_column]))

        builder = escher.Builder(
            model=model_json_path,
            map_name="iJO1366.Central metabolism",
            reaction_data=reaction_data,
        )
        builder.save_html(output_html_path)
        print(f"Escher HTML saved to {output_html_path}")
        return True
    except Exception as exc:
        print("Escher HTML generation failed in this environment:")
        print(f"  {type(exc).__name__}: {exc}")
        print("\nManual visualization instructions:")
        print("  a. Open https://escher.github.io/")
        print(f"  b. Load the iJO1366 model JSON: {model_json_path}")
        print("  c. Load the iJO1366 central metabolism map")
        print(f"  d. Load reaction data from: {flux_csv_path}")
        print("     (Escher menu: Data → Load reaction data)")
        return False

## 4. Exercise 1 — Load iJO1366 and inspect the objective

In [ ]:
base_model = load_model()

print(f"Reactions: {base_model.num_reactions}")
print(f"Metabolites: {base_model.num_metabolites}")
print(f"Genes: {base_model.num_genes}")

objective_rxn_id = get_objective_reaction_id(base_model)
print(f"\nActive objective (biomass) reaction: {objective_rxn_id}")
print(base_model.reactions.get_by_id(objective_rxn_id).reaction)

present, missing = validate_reactions(base_model, KEY_EXCHANGES)
print(f"\nKey exchanges present ({len(present)}): {present}")
if missing:
    print(f"Key exchanges missing ({len(missing)}): {missing}")

## 5. Wild-type simulations (Exercises 2 & 3)

We simulate four environmental conditions using **Flux Balance Analysis (FBA)**. Each simulation runs inside `with base_model:` so the original model stays unchanged.

In [ ]:
WT_CONDITIONS = [
    ("glucose_aerobic", CARBON_SOURCES["glucose"], 10, 20),
    ("glucose_anaerobic", CARBON_SOURCES["glucose"], 10, 0),
    ("glycerol_aerobic", CARBON_SOURCES["glycerol"], 10, 20),
    ("glycerol_anaerobic", CARBON_SOURCES["glycerol"], 10, 0),
]

results = {}

for condition_name, carbon_ex, carbon_uptake, o2_uptake in WT_CONDITIONS:
    with base_model:
        set_medium(base_model, carbon_ex, carbon_uptake=carbon_uptake, oxygen_uptake=o2_uptake)
        results[condition_name] = run_fba(
            base_model,
            condition_name,
            carbon_source=carbon_ex,
            oxygen_uptake=o2_uptake,
            knockout=None,
        )
    print(
        f"{condition_name:22s} | status={results[condition_name]['status']:10s} "
        f"| growth={results[condition_name]['objective_value']:.6f}"
    )

## 6. Reaction knockouts under glucose aerobic medium (Exercise 6)

In [ ]:
for rxn_id in MUTANT_REACTIONS:
    condition_name = f"mutant_{rxn_id}"
    with base_model:
        set_medium(
            base_model,
            CARBON_SOURCES["glucose"],
            carbon_uptake=10,
            oxygen_uptake=20,
        )
        success, message = apply_reaction_knockout(base_model, rxn_id)
        print(message)
        if success:
            results[condition_name] = run_fba(
                base_model,
                condition_name,
                carbon_source=CARBON_SOURCES["glucose"],
                oxygen_uptake=20,
                knockout=rxn_id,
            )
            print(
                f"  {condition_name:22s} | status={results[condition_name]['status']:10s} "
                f"| growth={results[condition_name]['objective_value']:.6f}"
            )

## 7. Growth summary and Escher export files

In [ ]:
# Growth summary table
summary_rows = []
for condition_name, result in results.items():
    summary_rows.append({
        "condition": result["condition"],
        "status": result["status"],
        "objective_value": result["objective_value"],
        "carbon_source": result["carbon_source"],
        "oxygen_uptake": result["oxygen_uptake"],
        "knockout": result["knockout"],
    })

growth_summary = pd.DataFrame(summary_rows)
growth_summary_path = Path("outputs/growth_summary.csv")
growth_summary.to_csv(growth_summary_path, index=False)
print(f"Saved {growth_summary_path}")
display(growth_summary)

# Escher-compatible flux CSV files
export_fluxes_for_escher(results)

## 8. Flux analysis (Exercise 4 & 7)

### Top 20 fluxes by absolute value (per condition)

In [ ]:
def print_top_fluxes(result, n=20):
    """Print the n largest absolute fluxes for one condition."""
    fluxes = result["fluxes"].copy()
    top = fluxes.reindex(fluxes.abs().sort_values(ascending=False).index).head(n)
    print(f"\nTop {n} fluxes — {result['condition']}:")
    for rxn_id, flux in top.items():
        print(f"  {rxn_id:20s}  {flux:12.4f}")
    return top


for condition_name, result in results.items():
    print_top_fluxes(result)

### Flux differences between conditions

In [ ]:
def flux_difference(results_dict, condition_a, condition_b):
    """Return a Series of flux differences: condition_a minus condition_b."""
    if condition_a not in results_dict or condition_b not in results_dict:
        raise KeyError(f"Missing condition: {condition_a} or {condition_b}")
    flux_a = results_dict[condition_a]["fluxes"]
    flux_b = results_dict[condition_b]["fluxes"]
    return flux_a.subtract(flux_b, fill_value=0.0)


comparisons = {
    "glucose_aerobic_minus_anaerobic": (
        "glucose_aerobic", "glucose_anaerobic"
    ),
    "glucose_aerobic_minus_glycerol_aerobic": (
        "glucose_aerobic", "glycerol_aerobic"
    ),
}

# Add mutant comparisons vs glucose_aerobic wild-type
for rxn_id in MUTANT_REACTIONS:
    col_name = f"mutant_{rxn_id}"
    if col_name in results:
        comparisons[f"glucose_aerobic_minus_{col_name}"] = (
            "glucose_aerobic", col_name
        )

flux_changes = {}
for label, (cond_a, cond_b) in comparisons.items():
    diff = flux_difference(results, cond_a, cond_b)
    flux_changes[label] = diff
    top_changed = diff.reindex(diff.abs().sort_values(ascending=False).index).head(10)
    print(f"\nTop 10 |flux change| — {label}:")
    for rxn_id, delta in top_changed.items():
        print(f"  {rxn_id:20s}  {delta:12.4f}")

# Export top changed reactions across all comparisons
top_change_rows = []
for label, diff in flux_changes.items():
    top_n = diff.reindex(diff.abs().sort_values(ascending=False).index).head(20)
    for rank, (rxn_id, delta) in enumerate(top_n.items(), start=1):
        top_change_rows.append({
            "comparison": label,
            "rank": rank,
            "reaction": rxn_id,
            "flux_difference": delta,
            "abs_flux_difference": abs(delta),
        })

top_flux_changes = pd.DataFrame(top_change_rows)
top_flux_changes_path = Path("outputs/top_flux_changes.csv")
top_flux_changes.to_csv(top_flux_changes_path, index=False)
print(f"\nSaved {top_flux_changes_path}")

## 9. Optional Escher HTML visualization (Exercise 5)

In [ ]:
escher_html_path = Path("escher_outputs/iJO1366_flux_visualization.html")
flux_csv_for_escher = Path("outputs/escher_glucose_aerobic_vs_anaerobic.csv")

make_escher_html(
    model_json_path=MODEL_PATH,
    flux_csv_path=flux_csv_for_escher,
    output_html_path=escher_html_path,
    condition_column="glucose_aerobic",
)

## 10. Download all outputs as a ZIP archive

In [ ]:
import zipfile

ZIP_NAME = "iJO1366_escher_exercise_outputs.zip"

if Path(ZIP_NAME).exists():
    Path(ZIP_NAME).unlink()

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["outputs", "escher_outputs", "data"]:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
        for file_path in folder_path.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=str(file_path))

print(f"Created {ZIP_NAME}")
print("\nExpected output files:")
expected = [
    "data/iJO1366.json",
    "outputs/growth_summary.csv",
    "outputs/escher_reaction_fluxes_wide.csv",
    "outputs/escher_glucose_aerobic_vs_anaerobic.csv",
    "outputs/escher_glucose_vs_glycerol_aerobic.csv",
    "outputs/escher_mutants_glucose_aerobic.csv",
    "outputs/top_flux_changes.csv",
    "escher_outputs/iJO1366_flux_visualization.html",
    ZIP_NAME,
]
for path_str in expected:
    status = "OK" if Path(path_str).exists() else "missing (optional)" if "html" in path_str else "MISSING"
    print(f"  [{status:16s}] {path_str}")

---

## Loading exported CSV files in Escher (Exercise 5)

Follow these steps to visualize your FBA results:

1. **Open Escher** in your browser: [https://escher.github.io/](https://escher.github.io/)

2. **Load the metabolic model**
   - Click **Model → Load COBRA Model**
   - Upload `data/iJO1366.json` (download from Colab if needed)

3. **Load a map**
   - Click **Map → Load Map**
   - Search for and select **iJO1366.Central metabolism**

4. **Load reaction flux data**
   - Click **Data → Load reaction data**
   - Upload one of the CSV files from `outputs/`:
     - `escher_glucose_aerobic_vs_anaerobic.csv` — compare aerobic vs anaerobic on glucose
     - `escher_glucose_vs_glycerol_aerobic.csv` — compare carbon sources under aerobic growth
     - `escher_mutants_glucose_aerobic.csv` — compare knockouts to wild-type
     - `escher_reaction_fluxes_wide.csv` — all conditions in one file

5. **Interpret the map**
   - Reaction arrows are colored and scaled by flux magnitude.
   - Positive flux (forward direction) and negative flux (reverse) are shown with different colors.
   - Click a reaction to inspect its ID, equation, and flux value.

6. **Compare conditions**
   - Escher can display two flux columns side by side when your CSV has multiple condition columns.
   - Use this to see how flux reroutes between aerobic/anaerobic growth, different carbon sources, or after gene knockouts.

> **Tip:** If the pre-built HTML in `escher_outputs/` was generated successfully, you can download and open it directly in a browser. Otherwise, use the manual steps above — they work on any machine with a web browser.